In [1]:
!pip install gdown
!pip install transformers sentencepiece accelerate
!pip install -q transformers datasets sacrebleu accelerate sentencepiece
from datasets import load_dataset, Dataset, concatenate_datasets
import random
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments
)
import sacrebleu

!pip install wandb
!pip install unbabel-comet

import wandb
wandb.login()


error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

ModuleNotFoundError: No module named 'datasets'

In [ ]:
# --- LOCAL CONFIGURATION ---
# 1. Install gdown if you haven't already: !pip install gdown
# 2. Get your Google Drive folder link (Right click folder -> Get Link -> Anyone with link)
# 3. Paste the link below

GDRIVE_LINK = "https://drive.google.com/drive/folders/1OgyH-YCKFNSVPvD41_Uf2GCG_-nEI5yw?usp=sharing"  # <--- PASTE LINK HERE
MODEL_NAME = "nllb-edu-en-ar-finetuned-v3"

import os
if GDRIVE_LINK and not os.path.exists(MODEL_NAME):
    print(f"Downloading model folder from Google Drive...")
    os.system(f'gdown --folder "{GDRIVE_LINK}" -O "{MODEL_NAME}"')
else:
    print("Model folder found or link not provided. Skipping download.")


In [ ]:
#UNPC Dataset
# Load UNPC (non-streaming for simplicity)
unpc = load_dataset("Helsinki-NLP/un_pc", "ar-en", split="train")

# Shuffle
unpc = unpc.shuffle(seed=42)

# Take 6500 samples
unpc = unpc.select(range(6_500))

# Convert to EN → AR format
def flip_unpc(example):
    return {
        "source": example["translation"]["en"],
        "target": example["translation"]["ar"]
    }

unpc = unpc.map(flip_unpc, remove_columns=unpc.column_names)

In [ ]:
#Tatoeba Dataset
tatoeba = load_dataset("ymoslem/Tatoeba-EN-AR", split="train")

# Shuffle
tatoeba = tatoeba.shuffle(seed=42)

# Take 3500 samples
tatoeba = tatoeba.select(range(3_500))

# Normalize column names
def clean_tatoeba(example):
    return {
        "source": example["English"],
        "target": example["Arabic"]
    }

tatoeba = tatoeba.map(clean_tatoeba, remove_columns=tatoeba.column_names)


In [ ]:
#Datasets split
tatoeba_split = tatoeba.train_test_split(test_size=0.09, seed=42)
unpc_split = unpc.train_test_split(test_size=0.09, seed=42)

unpc_train = unpc_split["train"]
unpc_test = unpc_split["test"]
tatoeba_train = tatoeba_split["train"]
tatoeba_test  = tatoeba_split["test"]

print(len(unpc_train))
print(len(unpc_test))
print(len(tatoeba_train))
print(len(tatoeba_test))

In [ ]:
#Combine the tow datasets
train_dataset = concatenate_datasets([unpc_train, tatoeba_train])
test_dataset  = concatenate_datasets([unpc_test, tatoeba_test])

# Final shuffle
train_dataset = train_dataset.shuffle(seed=42)
test_dataset  = test_dataset.shuffle(seed=42)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

train_dataset.save_to_disk("train_en_ar")
test_dataset.save_to_disk("test_en_ar")


In [ ]:
# --- LOCAL SETUP ---
import os
import gdown

# 1. Paste your Google Drive folder link here
GDRIVE_LINK = "https://drive.google.com/drive/folders/1OgyH-YCKFNSVPvD41_Uf2GCG_-nEI5yw?usp=sharing"  # PASTE YOUR LINK HERE (e.g., https://drive.google.com/drive/folders/...)
MODEL_DIR = "nllb-edu-en-ar-finetuned-v3"

def download_model(link, output_dir):
    if not os.path.exists(output_dir):
        print(f"Downloading model from Google Drive to {output_dir}...")
        # Use --folder flag for directories
        os.system(f'gdown --folder "{link}" -O "{output_dir}"')
    else:
        print(f"Model already exists at {output_dir}")

if GDRIVE_LINK:
    download_model(GDRIVE_LINK, MODEL_DIR)

MODEL_PATH = os.path.abspath(MODEL_DIR)

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print(f"Loading model from {MODEL_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)


In [ ]:
#Tokenization
MAX_LEN = 128

def tokenize(batch):
    return tokenizer(
        batch["source"],
        text_target=batch["target"],
        max_length=MAX_LEN,
        truncation=True,
        padding="max_length"
    )



In [ ]:
def filter_empty(example):
    return (
        example.get("source") is not None
        and example.get("target") is not None
        and example["source"].strip() != ""
        and example["target"].strip() != ""
    )

train_dataset = train_dataset.filter(filter_empty)
test_dataset  = test_dataset.filter(filter_empty)

train_dataset = train_dataset.map(
    tokenize,
    batched=True,
    remove_columns=train_dataset.column_names
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True,
    remove_columns=test_dataset.column_names
)




In [ ]:
#Data collector
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    padding=True
)


In [ ]:
#Hyperparameters
training_args = TrainingArguments(
    output_dir="./nllb-edu-en-ar",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    warmup_steps=500,
    num_train_epochs=5,
    #max_steps=10_000,
    fp16=True,
    logging_steps=200,
    save_steps=2000,
    save_total_limit=2,
    eval_strategy="no",
    report_to="wandb"
)



In [ ]:
#Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)



In [ ]:
#Finetuning
trainer.train()

In [ ]:
# from google.colab import drive # Skipped for local run
# drive.mount("/content/drive") # Skipped for local run

SAVE_DIR = "./nllb-edu-en-ar-finetuned-v4"
import os
os.makedirs(SAVE_DIR, exist_ok=True)

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)


In [ ]:
#BLUE Metric

# from google.colab import drive # Skipped for local run
# drive.mount("/content/drive") # Skipped for local run

#MODEL_PATH = "./nllb-edu-en-ar-finetuned"
MODEL_PATH = "./nllb-edu-en-ar-finetuned-v4"

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)
#The original model
# MODEL_NAME = "facebook/nllb-200-distilled-600M"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# SRC_LANG = "eng_Latn"
# TGT_LANG = "arb_Arab"

# tokenizer.src_lang = SRC_LANG
# tokenizer.tgt_lang = TGT_LANG
model.to(device)
model.eval()

def generate_translations(dataset, max_length=200):
    predictions = []
    references = []

    for example in dataset:
        inputs = tokenizer(
            example["source"],
            return_tensors="pt",
            truncation=True,
            max_length=128
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=max_length
            )

        pred = tokenizer.decode(outputs[0], skip_special_tokens=True)

        predictions.append(pred)
        references.append([example["target"]])  # BLEU expects list of refs

    return predictions, references

preds, refs = generate_translations(test_dataset)

import sacrebleu

bleu = sacrebleu.corpus_bleu(preds, refs)
print(f"BLEU score: {bleu.score:.2f}")

#chrf Metric
from sacrebleu import corpus_chrf

chrf = corpus_chrf(preds, refs)
print(f"chrF++: {chrf.score:.2f}")



In [ ]:
#COMET Metric
from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
model = load_from_checkpoint(model_path)
srcs = [example["source"] for example in test_dataset]

data = [
    {
        "src": s,
        "mt": p,
        "ref": r[0]
    }
    for s, p, r in zip(srcs, preds, refs)
]

scores = model.predict(data, batch_size=8, gpus=1)
print("COMET score:", sum(scores["scores"]) / len(scores["scores"]))


In [ ]:
from datasets import load_dataset
import torch
import sacrebleu
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
# from google.colab import drive # Skipped for local run
# drive.mount("/content/drive") # Skipped for local run
# Load OPUS-100 test split for Arabic ↔ English
test_dataset = load_dataset("opus100", "ar-en", split="test")

#MODEL_PATH = "./nllb-edu-en-ar-finetuned"
MODEL_PATH = "./nllb-edu-en-ar-finetuned-v4"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)

device = "cuda" if torch.cuda.is_available() else "cpu"

# MODEL_NAME = "facebook/nllb-200-distilled-600M"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# SRC_LANG = "eng_Latn"
# TGT_LANG = "arb_Arab"

# tokenizer.src_lang = SRC_LANG
# tokenizer.tgt_lang = TGT_LANG
model.eval()
model.to(device)
def generate_translations_opus(dataset, max_length=200):
    predictions = []
    references = []

    for example in dataset:
        src_text = example["translation"]["en"]  # English source
        tgt_text = example["translation"]["ar"]  # Arabic reference

        inputs = tokenizer(
            src_text,
            return_tensors="pt",
            truncation=True,
            max_length=128
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=max_length)

        pred = tokenizer.decode(outputs[0], skip_special_tokens=True)

        predictions.append(pred)
        references.append([tgt_text])  # BLEU expects list of references

    return predictions, references

preds, refs = generate_translations_opus(test_dataset)

bleu = sacrebleu.corpus_bleu(preds, refs)
print(f"OPUS-100 BLEU score: {bleu.score:.2f}")

#chrf Metric
from sacrebleu import corpus_chrf

chrf = corpus_chrf(preds, refs)
print(f"OPUS-100 chrF++: {chrf.score:.2f}")

In [ ]:
from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = [
    {
        "src": example["translation"]["en"],
        "mt": pred,
        "ref": ref[0]
    }
    for example, pred, ref in zip(test_dataset, preds, refs)
]

scores = comet_model.predict(
    data,
    batch_size=8,
    gpus=1 if torch.cuda.is_available() else 0
)

print("OPUS-100 COMET score:", sum(scores["scores"]) / len(scores["scores"]))


In [ ]:
#Initail preprocessing (Until getting the exact input from the STT model)
import re

# --- configuration: common fillers & artifact patterns ---
FILLERS = {
    "uh", "um", "you know", "like", "hmm", "erm", "ah", "oh"
}
ARTIFACT_PATTERN = re.compile(r"(\[.*?\]|<.*?>|\{.*?\})")  # [noise], <unk>, {laugh}, etc.
MULTI_SPACE = re.compile(r"\s+")

# helper regexes
REPEATED_WORDS = re.compile(r"\b(\w+)(?:\s+\1\b)+", flags=re.IGNORECASE)  # "is is" -> "is"
QUOTE_NORMALIZE = [
    (re.compile(r"[“”«»]"), '"'),
    (re.compile(r"[‘’]"), "'"),
]
UNICODE_SPACES = re.compile(r"[\u00A0\u2000-\u200B\u202F\u2060]")  # non-standard spaces

def remove_fillers(text):
    """Remove filler phrases listed in FILLERS (word-boundary aware)."""
    # Sort by length to remove multi-word fillers first (e.g., "you know")
    for filler in sorted(FILLERS, key=len, reverse=True):
        pattern = re.compile(r"\b" + re.escape(filler) + r"\b", flags=re.IGNORECASE)
        text = pattern.sub(" ", text)
    return text

def collapse_repeated_words(text):
    """Collapse repeated consecutive words: 'this is is good' -> 'this is good'"""
    # Apply repeatedly until no change (handles "a a a b" -> "a b")
    prev = None
    while prev != text:
        prev = text
        text = REPEATED_WORDS.sub(r"\1", text)
    return text

def ensure_sentence_punctuation(text):
    """Ensure sentence ends with ., ?, or ! — if not, add a period."""
    text = text.strip()
    if not text:
        return text
    if text[-1] not in ".!?":
        text = text + "."
    return text

def sentence_case(text):
    """Capitalize first character of sentence (simple)."""
    if not text:
        return text
    # preserve existing capitalization after first char
    return text[0].upper() + text[1:]

def preprocess_sentence(s: str) -> str:
    """Apply all preprocessing steps to a single string."""
    if s is None:
        return ""
    # 1. Normalize unicode spaces and replace with normal space
    s = UNICODE_SPACES.sub(" ", s)
    # 2. Remove artifacts like [noise], <unk>, {laughter}
    s = ARTIFACT_PATTERN.sub(" ", s)
    # 3. Normalize quotes
    for rx, repl in QUOTE_NORMALIZE:
        s = rx.sub(repl, s)
    # 4. Lowercase for safe filler removal (we will restore sentence case later)
    #    but keep a version to restore punctuation/casing decisions later
    s = s.strip()
    # 5. Remove filler words/disfluencies
    s = remove_fillers(s)
    # 6. Collapse repeated whitespace
    s = MULTI_SPACE.sub(" ", s).strip()
    # 7. Collapse repeated words
    s = collapse_repeated_words(s)
    # 8. Trim again
    s = s.strip()
    # 9. Ensure terminal punctuation and sentence case
    s = ensure_sentence_punctuation(s)
    s = sentence_case(s)
    return s

def preprocess_list(sentences):
    """Preprocess a list of sentences (returns new list)."""
    return [preprocess_sentence(s) for s in sentences]


# -------------------------
# Example usage
# -------------------------
if __name__ == "__main__":
    example_sentences = [
        "uh i will call you when i arrive",
        "this  is    is a   test   sentence",
        "[noise] the temperature is twenty three degrees",
        "you know i think it's okay",
        "hello world",  # lacks punctuation
        "<unk> can you repeat that please"
    ]

    cleaned = preprocess_list(example_sentences)
    for orig, clean in zip(example_sentences, cleaned):
        print("ORIG:", orig)
        print("CLEAN:", clean)
        print("---")


In [ ]:
from transformers import pipeline # <--- Add this line

#an example of how the input and the output will be and the pipline
text = [
    "Machine translation models are improving very quickly.",
    "This model supports more than 200 languages.",
    "Google Colab makes testing machine learning easy."
]
src_lang = "eng_Latn"
tgt_lang = "arb_Arab"

NMT = pipeline('translation', model=model, tokenizer=tokenizer, src_lang=src_lang, tgt_lang=tgt_lang, max_length=500)
translatedScript = NMT(text)
for i, j in zip(text, translatedScript):
  print(i)
  print(j['translation_text'])
  print()

ModuleNotFoundError: No module named 'transformers'

In [ ]:
# measure tokens per second metric
import time
import torch

model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to("cuda")

# pipeline
NMT = pipeline(
    "translation",
    model=model,
    tokenizer=tokenizer,
    src_lang="eng_Latn",
    tgt_lang="arb_Arab",
    device=0,
    max_length=200
)

sentences = [
    "Artificial intelligence is transforming the world rapidly.",
    "Machine translation quality has improved significantly in recent years.",
    "The weather today is sunny with a slight chance of rain.",
    "He enjoys reading books about history and science.",
    "This new smartphone features a powerful processor.",
    "The meeting will start at 10 AM sharp.",
    "She prepared a detailed presentation for the conference.",
    "Many students struggle with time management.",
    "The company announced a new investment in renewable energy.",
    "Healthy eating habits can improve overall well-being.",
    "The experiment was successful but needs further testing.",
    "Scientists discovered a new species of butterfly.",
    "The movie received excellent reviews from critics.",
    "He forgot his keys on the kitchen table.",
    "Traveling helps broaden one’s perspective.",
    "The teacher explained the lesson clearly.",
    "The bridge was closed for maintenance work.",
    "Online education has become increasingly popular.",
    "The doctor recommended getting more sleep.",
    "This program requires strong analytical skills.",
    "The restaurant is famous for its traditional dishes.",
    "She wrote a letter to her friend in another country.",
    "The library will remain open until midnight.",
    "He bought a gift for his sister’s birthday.",
    "Technology continues to evolve at a fast pace.",
    "Please submit your assignment by the end of the week.",
    "The museum hosts exhibitions from around the world.",
    "He trained for months to complete the marathon.",
    "The bus was delayed due to heavy traffic.",
    "They discussed the project during lunch.",
    "The city plans to build more public parks.",
    "The contract must be reviewed carefully.",
    "He lost his wallet while shopping.",
    "The application requires an internet connection.",
    "This musical instrument is over one hundred years old.",
    "The classroom was filled with excitement.",
    "The engineer fixed the machine in less than an hour.",
    "She is learning how to play the piano.",
    "The package is expected to arrive tomorrow.",
    "They decided to adopt a rescue dog.",
    "The festival attracts thousands of visitors each year.",
    "He enjoys hiking in the mountains.",
    "The software update includes important security improvements.",
    "The child asked many questions about space.",
    "The professor published a new research paper.",
    "The train left the station on time.",
    "The bakery sells fresh bread every morning.",
    "Sports can help develop teamwork and discipline.",
    "She saved money to buy a new laptop.",
    "The lake was calm and beautiful at sunset.",
    "He volunteered at the local community center.",
    "Students need to practice regularly to master new skills.",
    "The government introduced new economic policies.",
    "The phone battery lasts longer than previous models.",
    "He carefully followed the instructions on the label.",
    "They celebrated their anniversary with a special dinner.",
    "The network outage affected thousands of users.",
    "She wants to improve her language skills.",
    "The book contains valuable information about world history.",
    "The dog barked loudly at the stranger.",
    "He traveled to three countries last year.",
    "The store offers a wide range of products.",
    "She painted a beautiful landscape using watercolors.",
    "The hotel room had a stunning view of the ocean.",
    "The computer crashed due to a system error.",
    "He wears glasses to help him see more clearly.",
    "The flight was canceled because of the storm.",
    "Children need encouragement to build confidence.",
    "She bought ingredients to prepare dinner.",
    "The road was slippery after the rain.",
    "He listened to music while studying.",
    "They are planning to renovate their house next year.",
    "The scientist gave a lecture about climate change.",
    "He drinks coffee every morning before work.",
    "The new policy will take effect next month.",
    "She found a wallet and returned it to its owner.",
    "The hospital provides excellent medical care.",
    "He was excited to start his new job.",
    "The car needs regular maintenance to run smoothly.",
    "The bakery introduced a new type of pastry.",
    "She participated in the school’s art competition.",
    "The project deadline was extended by two weeks.",
    "They planted several trees in the garden.",
    "He discovered an old photograph in the attic.",
    "The teacher gave the students extra homework.",
    "She enjoys watching documentaries in her free time.",
    "The team worked together to solve the problem.",
    "He accidentally spilled water on his laptop.",
    "They moved to a new apartment last month.",
    "The athlete won a gold medal at the tournament.",
    "She asked for help with her homework.",
    "The company hired several new employees.",
    "He cooked a delicious meal for his family.",
    "The scientist analyzed the data carefully.",
    "She completed the task ahead of schedule.",
    "They visited a famous landmark during their trip.",
    "The movie was based on a true story.",
    "He set an alarm to wake up early.",
    "The concert was canceled at the last minute."
]

# Tokenize all sentences
tokenized = [tokenizer.encode(s, add_special_tokens=True) for s in sentences]

# Compute number of tokens per sentence
token_counts = [len(t) for t in tokenized]

# Minimum and maximum
min_tokens = min(token_counts)
max_tokens = max(token_counts)

print("Minimum tokens in a sentence:", min_tokens)
print("Maximum tokens in a sentence:", max_tokens)
# count input tokens
inputs_tok = tokenizer(sentences, padding=True, return_tensors="pt")
input_token_count = inputs_tok["input_ids"].numel()

torch.cuda.synchronize()
start = time.time()

outputs = NMT(sentences)

torch.cuda.synchronize()
end = time.time()
time_taken = end - start

# count output tokens by re-tokenizing
output_texts = [o["translation_text"] for o in outputs]
outputs_tok = tokenizer(output_texts, padding=True, return_tensors="pt")
output_token_count = outputs_tok["input_ids"].numel()

total_tokens = input_token_count + output_token_count
tokens_per_sec = total_tokens / time_taken

print("Input tokens:", input_token_count)
print("Output tokens:", output_token_count)
print("Total tokens:", total_tokens)
print("Time (sec):", time_taken)
print("Tokens/sec:", tokens_per_sec)


ModuleNotFoundError: No module named 'torch'

In [ ]:
#measure Latency per sentence metric

# Target language (for example Arabic)
tgt_lang = "arb_Arab"

translation_pipeline = pipeline(
    "translation",
    model=model,
    tokenizer=tokenizer,
    src_lang="eng_Latn",
    tgt_lang=tgt_lang,
    max_length=200
)

# Measure latency per sentence
latencies = []

for sentence in sentences:
    start_time = time.time()
    translation = translation_pipeline(sentence)[0]['translation_text']
    end_time = time.time()

    latency = end_time - start_time
    latencies.append(latency)

    print(f"Input: {sentence}")
    print(f"Translation: {translation}")
    print(f"Latency: {latency:.4f} seconds\n")

# Compute average latency
avg_latency = sum(latencies) / len(latencies)
print(f"Average latency per sentence: {avg_latency:.4f} seconds")